# Подготовка данных

Общая база для обоих продакшен-путей (desktop и edge):

1. **COCO-person** -> датасет базового обучения детектора (`person_dataset`).
2. **MOT20** -> кадры для domain adaptation детектора (`mot20_finetune`).

Результат - два YOLO-датасета в `data/`, которые читает `detector_training.ipynb`.

In [ ]:
import shutil
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import yaml
from ultralytics.data.utils import check_det_dataset
import configparser

ROOT = Path().resolve().parent
DATA_DIR = ROOT / "data"
PERSON_DATASET = DATA_DIR / "person_dataset"
PERSON_YAML = DATA_DIR / "person_dataset.yaml"
PERSON_CLASS_COCO = 0
LAPLACIAN_THRESH = 100       
VAL_SPLIT = 0.15
SEED = 42
MOT20_DIR = DATA_DIR / "MOT20" / "MOT20" / "train"
MOT20_DATASET = DATA_DIR / "mot20_finetune"
MOT20_YAML = MOT20_DATASET / "dataset.yaml"
MOT20_TRAIN_SEQS = ["MOT20-02", "MOT20-03", "MOT20-05"]
MOT20_VAL_SEQS = ["MOT20-01"]
FRAME_STEP = 5              
MIN_VIS = 0.25                
MIN_BBOX_PX = 10
np.random.seed(SEED)
DATA_DIR

## 1. COCO-person -> датасет детектора

In [ ]:
def find_person_images(images_dir, labels_dir):
    """Кадры с хотя бы одним bbox person (класс 0) → DataFrame(image, label, n_persons)."""
    rows = []
    for lbl in sorted(labels_dir.glob("*.txt")):
        img = next((images_dir / (lbl.stem + e) for e in (".jpg", ".jpeg", ".png") if (images_dir / (lbl.stem + e)).exists()), None)
        if img is None or lbl.stat().st_size == 0:
            continue
        n = sum(1 for ln in lbl.read_text().split("\n") if ln and int(ln.split()[0]) == PERSON_CLASS_COCO)
        if n:
            rows.append({"image": img, "label": lbl, "n_persons": n})
    return pd.DataFrame(rows)


coco = Path(check_det_dataset("coco128.yaml")["path"])
df = find_person_images(coco / "images" / "train2017", coco / "labels" / "train2017")
df.head()

In [ ]:
def laplacian_var(path):
    """Дисперсия Лапласиана — мера резкости (ниже порога считаем смазом)."""
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    return 0.0 if img is None else float(cv2.Laplacian(img, cv2.CV_64F).var())

df["sharpness"] = df["image"].map(laplacian_var)
df = df[df["sharpness"] >= LAPLACIAN_THRESH].reset_index(drop=True)
df.shape

In [ ]:
def export_person_split(rows, split, out):
    """Копирует кадры и пишет YOLO-метки, оставляя только person (класс → 0)."""
    img_dir, lbl_dir = out / "images" / split, out / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    for r in rows.itertuples():
        shutil.copy2(r.image, img_dir / r.image.name)
        person = ["0 " + " ".join(ln.split()[1:]) for ln in r.label.read_text().split("\n") if ln and int(ln.split()[0]) == PERSON_CLASS_COCO]
        (lbl_dir / r.label.name).write_text("\n".join(person))


df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
n_val = int(len(df) * VAL_SPLIT)

if PERSON_DATASET.exists():
    shutil.rmtree(PERSON_DATASET)
export_person_split(df.iloc[n_val:], "train", PERSON_DATASET)
export_person_split(df.iloc[:n_val], "val", PERSON_DATASET)

PERSON_YAML.write_text(yaml.dump({"path": str(PERSON_DATASET), "train": "images/train", "val": "images/val", "nc": 1, "names": ["person"]}, allow_unicode=True))
print(PERSON_YAML.read_text())

## 2. MOT20 -> датасет domain adaptation

Кадры плотных уличных сцен с GT-разметкой людей. Каждый `FRAME_STEP`-й кадр,
только видимые боксы (`MIN_VIS`, `MIN_BBOX_PX`), single-class person.

In [ ]:
def load_mot_gt(gt_path, img_w, img_h):
    """GT MOT20 → YOLO-боксы {frame_id: [[cx,cy,w,h], ...]}"""
    gt = {}
    for line in gt_path.read_text().strip().splitlines():
        p = line.split(",")
        conf, cls = int(p[6]), int(p[7])
        vis = float(p[8]) if len(p) > 8 else 1.0
        if conf == 0 or cls != 1 or vis < MIN_VIS:
            continue
        x, y, w, h = map(float, p[2:6])
        if w < MIN_BBOX_PX or h < MIN_BBOX_PX:
            continue
        gt.setdefault(int(p[0]), []).append([
            float(np.clip((x + w / 2) / img_w, 0, 1)), float(np.clip((y + h / 2) / img_h, 0, 1)),
            float(np.clip(w / img_w, 0, 1)), float(np.clip(h / img_h, 0, 1))])
    return gt

def export_mot_split(seqs, split, out):
    """Каждый FRAME_STEP-й кадр последовательностей + YOLO-метки → out/{split}. Вернёт число кадров с боксами."""
    img_dir, lbl_dir = out / "images" / split, out / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    labeled = 0
    for seq in seqs:
        seq_dir = MOT20_DIR / seq
        ini = configparser.ConfigParser()
        ini.read(seq_dir / "seqinfo.ini")
        w, h = int(ini["Sequence"]["imWidth"]), int(ini["Sequence"]["imHeight"])
        gt = load_mot_gt(seq_dir / "gt" / "gt.txt", w, h)
        for ff in sorted((seq_dir / "img1").glob("*.jpg"))[::FRAME_STEP]:
            shutil.copy2(ff, img_dir / f"{seq}_{ff.stem}.jpg")
            boxes = gt.get(int(ff.stem), [])
            if boxes:
                (lbl_dir / f"{seq}_{ff.stem}.txt").write_text("\n".join(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}" for cx, cy, bw, bh in boxes))
                labeled += 1
    return labeled

In [ ]:
MOT20_DATASET.mkdir(parents=True, exist_ok=True)
n_train = export_mot_split(MOT20_TRAIN_SEQS, "train", MOT20_DATASET)
n_val = export_mot_split(MOT20_VAL_SEQS, "val", MOT20_DATASET)

MOT20_YAML.write_text(yaml.dump({"path": str(MOT20_DATASET.resolve()), "train": "images/train", "val": "images/val", "nc": 1, "names": ["person"]}, allow_unicode=True))
{"train_labeled_frames": n_train, "val_labeled_frames": n_val, "yaml": str(MOT20_YAML)}